# 08 - Beta-beta comparison

Compares effect sizes between two analyses - typically the cross-sectional
(log OR) and survival (log HR) meta-analyses on the same variants - and reports
correlation, regression slope and sign concordance.

Both inputs need the columns `SNP`, `BETA`, `SE` and `P`;
`rename_summary_stats()` maps the METAL/LocusZoom names onto them.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gwas import config as cfg
from gwas import betabeta as bb

pd.options.mode.chained_assignment = None

cfg.describe()

In [ ]:
# Mount the release and working buckets with gcsfuse (run once per VM).
for cmd in cfg.mount_commands():
    print(cmd)

## Variants to compare

In [ ]:
# Curated variant table: one row per reported variant, with the analysis it
# came from and the nearest gene.
curated = pd.read_csv(cfg.prep_path('meta', 'forest_plots', 'curated_variants.csv'))
display(curated)

EXCLUDE_GENES = []    # e.g. pseudogenes that are not worth plotting
target_snps = curated[curated.Variant.isin(['HY3_LR', 'HY3_CPH'])
                      & ~curated.Gene.isin(EXCLUDE_GENES)].dbSNP_ID.tolist()
print(f'{len(target_snps)} variants')

## Load the two analyses

These are the annotated METAL outputs written by `06_clean_results.ipynb`;
`rsid` becomes `SNP` so the two can be merged on a common identifier.

In [ ]:
def load_meta(analysis):
    path = (cfg.prep_path('meta', 'METAL', analysis, 'locuszoom', 'combined')
            / f'METAL_{analysis}.tsv.gz')
    df = pd.read_csv(path, compression='gzip', sep='\t')
    return bb.rename_summary_stats(df)


surv = load_meta('SURV')
cs_disc = load_meta('CS_disc')

print(sorted(set(surv.columns) & {'SNP', 'BETA', 'SE', 'P', 'A1', 'A2', 'EAF'}))
print(f'SURV: {len(surv):,} variants | CS_disc: {len(cs_disc):,} variants')

## Compare

`annotate_all` labels every plotted variant; with a long list use
`annotate_discordant=N` instead to label only the N variants whose effects
disagree most.

In [ ]:
out_dir = cfg.prep_path('meta', 'beta_beta')
out_dir.mkdir(parents=True, exist_ok=True)

fig, ax, metrics, merged = bb.compare_effect(
    surv, cs_disc,
    snplist=target_snps,
    label=['SURV', 'CS_disc', 'Both', 'None'],
    annotate_all=True,
    plot_error_bars=True,
    save=out_dir / 'SURV_vs_CS_disc_hits.png',
    return_data=True,
)
plt.show()
display(pd.Series(metrics))

## Genome-wide comparison

The same comparison over every shared variant rather than the reported hits.
Error bars and labels are off b/c of computational cost.

In [ ]:
fig, ax, metrics = bb.compare_effect(
    surv, cs_disc,
    label=['SURV', 'CS_disc', 'Both', 'None'],
    annotate_discordant=5,
    plot_error_bars=False,
    s=6,
    save=out_dir / 'SURV_vs_CS_disc_all.png',
)
plt.show()
display(pd.Series(metrics))

## Duplicated variants

`compare_effect` keeps the lowest-p row per SNP

In [ ]:
for df, name in ((surv, 'SURV'), (cs_disc, 'CS_disc')):
    bb.report_duplicates(df[df.SNP.isin(target_snps)], name)